# Five-Year GO Impact Report

This tutorial treats GO temporal change analysis as a configured convenience report. The report command runs the two enrichments, performs the threshold-crossing comparison, writes Parquet plus YAML/JSON metadata, and records the timing and parameters used. The notebook then inspects those report artifacts with DuckDB instead of rebuilding the workflow step by step.

In [1]:
%%bash
set -euo pipefail
cd "$(git rev-parse --show-toplevel)"
uv run --project python/genesets-workflows genesets-workflows go-impact evals/go_impact_5y_expression500.yaml

Wrote notebooks/generated/go_impact_5y_expression500/summary.yaml
Diff rows: gained_significant=2337

8, lost_significant=954, shared_significant=2760
Timings: old=3.925s, new=5.114s, compare=0.174s, to

tal=9.744s


In [2]:
import json
from pathlib import Path

import duckdb
import pandas as pd

from notebooks.display_utils import display_table, metric_cards, repo_root
from notebooks.go_filters import go_subset_descendants

repo = repo_root()
report_dir = repo / "notebooks/generated/go_impact_5y_expression500"
report = json.loads((report_dir / "summary.json").read_text())
params = report["report_parameters"]
rank_scope = params["post_processing"]["ranked_term_scope"]
diff_path = repo / report["outputs"]["diff_results"]
cutoff = params["compare"]["p_adjust_cutoff"]
max_ranked_target_size = int(rank_scope["max_target_size"])
scope_snapshot = params["snapshots"][rank_scope["snapshot"]]
go_dir = Path(scope_snapshot["dir"])
if not go_dir.is_absolute():
    go_dir = repo / go_dir
term_scope = go_subset_descendants(go_dir, rank_scope["subset"])
allowed_terms = term_scope.allowed_terms if rank_scope.get("include_descendants", True) else term_scope.subset_terms
term_filter_df = pd.DataFrame({"target_id": sorted(allowed_terms)})
con = duckdb.connect()
con.register("go_term_filter_df", term_filter_df)
con.execute("CREATE OR REPLACE TEMP TABLE go_term_filter AS SELECT * FROM go_term_filter_df")

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE diff_scored AS
    WITH params AS (SELECT ?::DOUBLE AS cutoff),
    scored AS (
      SELECT
        d.*,
        CASE
          WHEN d.left_p_adjust IS NULL THEN -log10(params.cutoff)
          WHEN d.left_p_adjust <= 0 THEN 320.0
          ELSE -log10(d.left_p_adjust)
        END AS left_score,
        CASE
          WHEN d.right_p_adjust IS NULL THEN -log10(params.cutoff)
          WHEN d.right_p_adjust <= 0 THEN 320.0
          ELSE -log10(d.right_p_adjust)
        END AS right_score
      FROM read_parquet(?) AS d
      CROSS JOIN params
    )
    SELECT
      *,
      right_score - left_score AS signed_delta_score,
      abs(right_score - left_score) AS abs_delta_score
    FROM scored
    """,
    [cutoff, str(diff_path)],
)

counts = report["result_counts"]
timings = report["timings"]
metric_cards([
    ("expression gene sets", f"{report['query_sets']['selected_count']:,}"),
    ("new significant rows", f"{counts['new']['rows']:,}"),
    ("gained calls", f"{counts['diff'].get('gained_significant', 0):,}"),
    ("lost calls", f"{counts['diff'].get('lost_significant', 0):,}"),
    ("rankable GO terms", f"{len(allowed_terms):,}"),
    ("total runtime", f"{timings['total_seconds']:.2f}s"),
])

In [3]:
parameter_rows = [
    {"parameter": "config", "value": report["config_path"]},
    {"parameter": "query source", "value": params["query_sets"]["source_dir"]},
    {"parameter": "query limit", "value": f"{params['query_sets']['limit']:,}"},
    {"parameter": "left snapshot", "value": f"{params['snapshots']['left']['label']} ({params['snapshots']['left']['annotation_variant']})"},
    {"parameter": "right snapshot", "value": f"{params['snapshots']['right']['label']} ({params['snapshots']['right']['annotation_variant']})"},
    {"parameter": "min overlap", "value": params["statistics"]["min_overlap"]},
    {"parameter": "adjusted p cutoff", "value": params["compare"]["p_adjust_cutoff"]},
    {"parameter": "ranked term scope", "value": f"{rank_scope['subset']} descendants from {rank_scope['snapshot']} snapshot"},
    {"parameter": "max ranked term size", "value": f"{max_ranked_target_size:,}"},
]
display_table(pd.DataFrame(parameter_rows), caption="Report parameters", max_rows=None)

display_table(
    pd.DataFrame([{"artifact": key, "path": value} for key, value in report["outputs"].items()]),
    caption="Generated artifacts",
    max_rows=None,
)

parameter,value
config,evals/go_impact_5y_expression500.yaml
query source,evals/expression_like/generated/msigdb_gse_500
query limit,500
left snapshot,2021-05-01 (all)
right snapshot,2026-03-25 (all)
min overlap,2
adjusted p cutoff,0.05
ranked term scope,goslim_generic descendants from right snapshot
max ranked term size,"1,000"


artifact,path
diff_metadata,notebooks/generated/go_impact_5y_expression500/go_2021_05_01_vs_go_2026_03_25_expression500.diff.yaml
diff_results,notebooks/generated/go_impact_5y_expression500/go_2021_05_01_vs_go_2026_03_25_expression500.diff.parquet
new_results,notebooks/generated/go_impact_5y_expression500/go_2026_03_25_expression500.parquet
old_results,notebooks/generated/go_impact_5y_expression500/go_2021_05_01_expression500.parquet
summary_json,notebooks/generated/go_impact_5y_expression500/summary.json
summary_yaml,notebooks/generated/go_impact_5y_expression500/summary.yaml


In [4]:
timing_rows = pd.DataFrame([
    {"stage": f"select {params['query_sets']['limit']:,} expression sets", "seconds": timings["select_queries_seconds"]},
    {"stage": f"{params['snapshots']['left']['label']} GO enrichment", "seconds": timings["old_matrix_seconds"]},
    {"stage": f"{params['snapshots']['right']['label']} GO enrichment", "seconds": timings["new_matrix_seconds"]},
    {"stage": "threshold-crossing compare", "seconds": timings["compare_seconds"]},
    {"stage": "total report", "seconds": timings["total_seconds"]},
])
display_table(
    timing_rows,
    caption="Report timings on this laptop",
    max_rows=None,
    formatters={"seconds": lambda value: f"{value:.3f}"},
)

stage,seconds
select 500 expression sets,0.006
2021-05-01 GO enrichment,3.925
2026-03-25 GO enrichment,5.114
threshold-crossing compare,0.174
total report,9.744


In [5]:
class_counts = con.execute(
    """
    SELECT
      class,
      count(*) AS row_count,
      count(DISTINCT query_id) AS affected_query_sets,
      count(DISTINCT target_id) AS affected_go_terms
    FROM diff_scored
    GROUP BY class
    ORDER BY row_count DESC
    """,
).df()
display_table(class_counts, caption="Significance status from 2021 to 2026", max_rows=None)

class,row_count,affected_query_sets,affected_go_terms
gained_significant,23378,500,1067
shared_significant,2760,204,494
lost_significant,954,128,235


For each high-impact expression signature, the term columns identify the GO term with the largest signed change in `-log10(Bonferroni adjusted p)` after applying the configured ranked-term scope. In this report, candidates are current GO terms in `goslim_generic` or descendants of a `goslim_generic` term, with at most 1,000 annotated genes in the relevant snapshot. Positive values mean stronger enrichment in 2026; negative values mean weaker enrichment. For gained/lost calls, the non-significant side is anchored at the report cutoff (`0.05`) because this report stores significant-only result tables.

This is an inclusion slim plus a size guard. The complementary antislim is the same idea with the `go_term_filter` join inverted, which is useful when the goal is to explicitly remove a known broad region of the ontology from a report.

In [6]:
by_query = con.execute(
    """
    WITH counts AS (
      SELECT
        query_id,
        CAST(sum(CASE WHEN class = 'gained_significant' THEN 1 ELSE 0 END) AS BIGINT) AS gained_calls,
        CAST(sum(CASE WHEN class = 'lost_significant' THEN 1 ELSE 0 END) AS BIGINT) AS lost_calls,
        count(*) AS total_crossing_calls
      FROM diff_scored
      WHERE class IN ('gained_significant', 'lost_significant')
      GROUP BY query_id
    ),
    crossing_ranked AS (
      SELECT
        query_id,
        target_id,
        target_name,
        class,
        signed_delta_score,
        abs_delta_score,
        row_number() OVER (PARTITION BY query_id ORDER BY abs_delta_score DESC, target_id) AS rank
      FROM diff_scored
      JOIN go_term_filter USING (target_id)
      WHERE class IN ('gained_significant', 'lost_significant')
        AND coalesce(right_target_size, left_target_size) <= ?
    ),
    overall_ranked AS (
      SELECT
        query_id,
        target_id,
        target_name,
        class,
        signed_delta_score,
        abs_delta_score,
        row_number() OVER (PARTITION BY query_id ORDER BY abs_delta_score DESC, target_id) AS rank
      FROM diff_scored
      JOIN go_term_filter USING (target_id)
      WHERE coalesce(right_target_size, left_target_size) <= ?
    )
    SELECT
      c.query_id,
      c.gained_calls,
      c.lost_calls,
      c.total_crossing_calls,
      crossing.target_id || ' ' || crossing.target_name AS largest_crossing_term,
      crossing.class AS crossing_status,
      crossing.signed_delta_score AS largest_crossing_delta,
      overall_top.target_id || ' ' || overall_top.target_name AS largest_overall_term,
      overall_top.class AS overall_status,
      overall_top.signed_delta_score AS largest_overall_delta
    FROM counts AS c
    LEFT JOIN crossing_ranked AS crossing
      ON c.query_id = crossing.query_id AND crossing.rank = 1
    LEFT JOIN overall_ranked AS overall_top
      ON c.query_id = overall_top.query_id AND overall_top.rank = 1
    ORDER BY c.total_crossing_calls DESC, c.gained_calls DESC
    LIMIT 25
    """,
    [max_ranked_target_size, max_ranked_target_size],
).df()
for status_column in ["crossing_status", "overall_status"]:
    by_query[status_column] = by_query[status_column].str.replace("_significant", "", regex=False)
display_table(
    by_query,
    caption="Expression signatures with the most threshold crossings; largest terms use the configured ranked-term scope",
    max_rows=25,
    formatters={
        "largest_crossing_delta": lambda value: f"{value:+.2f}",
        "largest_overall_delta": lambda value: f"{value:+.2f}",
    },
)

query_id,gained_calls,lost_calls,total_crossing_calls,largest_crossing_term,crossing_status,largest_crossing_delta,largest_overall_term,overall_status,largest_overall_delta
GSE29618_BCELL_VS_MDC_DAY7_FLU_VACCINE_DN,177,31,208,GO:0002366 leukocyte activation involved in immune response,lost,-7.35,GO:0030141 secretory granule,shared,+8.37
GSE29618_PDC_VS_MDC_DAY7_FLU_VACCINE_DN,178,27,205,GO:0002366 leukocyte activation involved in immune response,lost,-9.80,GO:0002252 immune effector process,shared,-11.55
GSE29618_PDC_VS_MDC_DN,167,32,199,GO:0002366 leukocyte activation involved in immune response,lost,-17.21,GO:0002366 leukocyte activation involved in immune response,lost,-17.21
GSE29618_MONOCYTE_VS_PDC_UP,158,26,184,GO:0042119 neutrophil activation,lost,-22.09,GO:0002274 myeloid leukocyte activation,shared,-22.42
GSE29618_BCELL_VS_MONOCYTE_DN,143,26,169,GO:0002283 neutrophil activation involved in immune response,lost,-26.45,GO:0002274 myeloid leukocyte activation,shared,-26.64
GSE28726_NAIVE_VS_ACTIVATED_CD4_TCELL_DN,133,36,169,GO:0060205 cytoplasmic vesicle lumen,gained,+4.92,GO:0000278 mitotic cell cycle,shared,+8.56
GSE29164_UNTREATED_VS_CD8_TCELL_TREATED_MELANOMA_DAY7_UP,138,21,159,GO:0006954 inflammatory response,gained,+4.89,GO:0006954 inflammatory response,gained,+4.89
GSE29618_MONOCYTE_VS_MDC_UP,127,27,154,GO:0042119 neutrophil activation,lost,-32.05,GO:0042119 neutrophil activation,lost,-32.05
GSE29618_BCELL_VS_MDC_DN,117,35,152,GO:0002252 immune effector process,lost,-13.79,GO:0002252 immune effector process,lost,-13.79
GSE28726_NAIVE_CD4_TCELL_VS_NAIVE_VA24NEG_NKTCELL_UP,123,22,145,GO:0006281 DNA repair,gained,+5.33,GO:0000278 mitotic cell cycle,shared,+13.66


The next table asks which GO terms changed status across the largest number of expression signatures. It reuses the configured `goslim_generic` descendant scope and adds a stricter display guard of at most 500 genes so the table stays biologically interpretable.

In [7]:
by_specific_term = con.execute(
    """
    SELECT
      class,
      target_id,
      target_name,
      coalesce(right_target_size, left_target_size) AS target_size,
      count(DISTINCT query_id) AS affected_query_sets
    FROM diff_scored
    JOIN go_term_filter USING (target_id)
    WHERE class IN ('gained_significant', 'lost_significant')
      AND coalesce(right_target_size, left_target_size) <= 500
    GROUP BY class, target_id, target_name, target_size
    ORDER BY affected_query_sets DESC, target_size ASC, target_id
    LIMIT 30
    """,
).df()
display_table(by_specific_term, caption="Most repeatedly affected specific GO terms within the configured ranked-term scope", max_rows=30)

class,target_id,target_name,target_size,affected_query_sets
gained_significant,GO:0009897,external side of plasma membrane,392,25
gained_significant,GO:0046649,lymphocyte activation,414,21
gained_significant,GO:0002764,immune response-regulating signaling pathway,468,20
gained_significant,GO:0002521,leukocyte differentiation,339,18
gained_significant,GO:1903131,mononuclear cell differentiation,294,16
gained_significant,GO:0006954,inflammatory response,447,16
gained_significant,GO:0002768,immune response-regulating cell surface receptor signaling pathway,380,15
gained_significant,GO:0030098,lymphocyte differentiation,223,14
gained_significant,GO:0002757,immune response-activating signaling pathway,398,14
gained_significant,GO:0006897,endocytosis,429,14


In [8]:
top_crossings = con.execute(
    """
    SELECT
      class,
      query_id,
      target_id,
      target_name,
      signed_delta_score,
      abs_delta_score,
      left_p_adjust,
      right_p_adjust,
      left_overlap,
      right_overlap,
      left_target_size,
      right_target_size
    FROM diff_scored
    JOIN go_term_filter USING (target_id)
    WHERE class IN ('gained_significant', 'lost_significant')
      AND coalesce(right_target_size, left_target_size) <= ?
    ORDER BY abs_delta_score DESC, target_id
    LIMIT 30
    """,
    [max_ranked_target_size],
).df()
top_crossings["class"] = top_crossings["class"].str.replace("_significant", "", regex=False)
display_table(
    top_crossings,
    caption="Largest individual threshold-crossing changes within the configured ranked-term scope",
    max_rows=30,
    formatters={
        "signed_delta_score": lambda value: f"{value:+.2f}",
        "abs_delta_score": lambda value: f"{value:.2f}",
        "left_p_adjust": lambda value: "" if pd.isna(value) else f"{value:.3e}",
        "right_p_adjust": lambda value: "" if pd.isna(value) else f"{value:.3e}",
    },
)

class,query_id,target_id,target_name,signed_delta_score,abs_delta_score,left_p_adjust,right_p_adjust,left_overlap,right_overlap,left_target_size,right_target_size
lost,GSE29618_MONOCYTE_VS_MDC_DAY7_FLU_VACCINE_UP,GO:0042119,neutrophil activation,-32.99,32.99,5.068e-35,,57,,498,
lost,GSE29618_MONOCYTE_VS_MDC_DAY7_FLU_VACCINE_UP,GO:0036230,granulocyte activation,-32.75,32.75,8.905e-35,,57,,503,
lost,GSE29618_MONOCYTE_VS_MDC_DAY7_FLU_VACCINE_UP,GO:0043312,neutrophil degranulation,-32.59,32.59,1.292e-34,,56,,482,
lost,GSE29618_MONOCYTE_VS_MDC_DAY7_FLU_VACCINE_UP,GO:0002274,myeloid leukocyte activation,-32.32,32.32,2.383e-34,,60,,589,
lost,GSE29618_MONOCYTE_VS_MDC_DAY7_FLU_VACCINE_UP,GO:0002283,neutrophil activation involved in immune response,-32.29,32.29,2.568e-34,,56,,488,
lost,GSE29618_MONOCYTE_VS_MDC_UP,GO:0042119,neutrophil activation,-32.05,32.05,4.497e-34,,56,,498,
lost,GSE29618_MONOCYTE_VS_MDC_DAY7_FLU_VACCINE_UP,GO:0002446,neutrophil mediated immunity,-31.90,31.90,6.323e-34,,56,,496,
lost,GSE29618_MONOCYTE_VS_MDC_UP,GO:0036230,granulocyte activation,-31.81,31.81,7.815e-34,,56,,503,
lost,GSE29618_MONOCYTE_VS_MDC_DAY7_FLU_VACCINE_UP,GO:0002275,myeloid cell activation involved in immune response,-31.57,31.57,1.360e-33,,57,,528,
lost,GSE29618_MONOCYTE_VS_MDC_DAY7_FLU_VACCINE_UP,GO:0043299,leukocyte degranulation,-31.37,31.37,2.125e-33,,56,,507,
